In [ ]:
# Three_channel_image

> Three channel image creation 

In [ ]:
#| default_exp three_channel_dataset

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
# | export
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from fastcore.all import *
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import shutil
from functools import partial
from typing import Union, List, Tuple
from sklearn.model_selection import train_test_split
import json
import yaml
import albumentations as A
import os

In [ ]:
#| export
import tensorflow as tf

In [ ]:
#| export
import matplotlib as mpl

In [ ]:
#| hide
from private_easy_pin_detection.core import *

In [ ]:
#mpl.rcParams["image.cmap"] = "gray"

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| hide
im2_path = Path(
    r"/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/Three_channel_image/im2"
)
im1_path = Path(
    r"/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/Three_channel_image/im1"
)
msk_path = Path(
    r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/Three_channel_image/im2_masks/time_11_18_24_val_frGrnd0.9470_epoch_185.h5'
               )

In [ ]:
#| hide
images = im2_path.ls()
sample_img2 = images[0]
image_name = Path(sample_img2).name
sample_img1 = Path(
    im1_path, 
    image_name.replace('image2', 'image1'))

In [ ]:
#| hide
img1 = read_img(sample_img1)
img2 = read_img(sample_img2)
three_c_img = cv2.merge([img2, img2, img1])

In [ ]:
#| hide
show_(three_c_img)

In [ ]:
#| export
def create_sum(path):

    df = pd.DataFrame()
    for idx, i in enumerate(tqdm(Path(path).ls(), total=len(path.ls()))):
        name_ = get_m_name(Path(i).name)
        df.loc[idx, 'rec_name'] = name_
        df.loc[idx, 'fp'] = i
        df.loc[idx, 'fn'] = Path(i).name
        
    return (
        df.assign(
            pin_type=lambda df_: df_['rec_name'].map(get_pin_type()),
        )
    )

In [ ]:
#| export
def from_trn_copy(
        df_trn: pd.DataFrame,
        im1_path: Union[str, Path],
        im2_path: Union[str, Path],
        msk_path: Union[str, Path],
        trn:bool=True
        ):
    'From trn dataframe copy images to trn folder normally base as im2_path'
    base_path = Path(im2_path).parent
    if trn:
        #################################
        trn_path = Path(f'{base_path}/trn')
        trn_im2_path = Path(trn_path, 'im2')
        trn_im1_path = Path(trn_path, 'im1')
        trn_msk_path = Path(trn_path, 'msk')
    else:
        trn_path = Path(f'{base_path}/val')
        trn_im2_path = Path(trn_path, 'im2')
        trn_im1_path = Path(trn_path, 'im1')
        trn_msk_path = Path(trn_path, 'msk')

    ####################################
    trn_im2_path.mkdir(exist_ok=True, parents=True)
    trn_im1_path.mkdir(exist_ok=True, parents=True)
    trn_msk_path.mkdir(exist_ok=True, parents=True)

    for i in tqdm(df_trn['fn'],total=len(df_trn)):
        im1_name = i.replace('image2', 'image1')
        im1_p = Path(im1_path, im1_name)
        im2_p = Path(im2_path, i)
        shutil.copyfile(im1_p, trn_im1_path/im1_name)
        shutil.copyfile(im2_p, trn_im2_path/i)
        shutil.copyfile(f'{msk_path}/{i}', f'{trn_msk_path}/{i}')

    print(f'Done copying')

# Create dataset from Folder

In [ ]:
def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print('{}{}'.format(subindent, f))

#list_files(f'{base_path}')

In [ ]:
#| export
def check_common_names(
        im1_path:Union[str, Path],
        im2_path:Union[str, Path]):

    print(f' Number of im1 images = {len(Path(im1_path).ls())}')
    print(f' Number of im2 images = {len(Path(im2_path).ls())}')
    replace_name = np.vectorize(lambda x: x.replace('image1', 'image2'))
    im1_names = get_name_(Path(im1_path).ls())
    im2_names = replace_name(im1_names)
    common_files = list(set(get_name_(im2_path.ls())).intersection(set(im2_names)))
    print(f' Number of common files = {len(common_files)}')
    return common_files

In [ ]:
#| export
def create_sinlge_image(
        im1_path:Union[str, Path],
        im2_path:Union[str, Path],
        im_save_path:Union[str, Path],# where three channel image will be saved
        second_channel:str='im2',# if im1, then [im2, im1, im1] else [im2, im2, im1]
        ):
    ' Combine both images to create a three channel image and save them'

    Path(im_save_path).mkdir(exist_ok=True, parents=True)

    common_im2_files = check_common_names(im1_path, im2_path)

    for i in tqdm(common_im2_files, total=len(common_im2_files)):
        im1_name = i.replace('image2', 'image1')
        im1_path_ = Path(im1_path, im1_name)
        img2 = read_img(Path(im2_path, i), gray=True, cv=True)
        img1 = read_img(im1_path_, gray=True, cv=True)
        if second_channel == 'im2':
            three_c_img = cv2.merge([img1, img2, img2])
        else:
            three_c_img = cv2.merge([img1, img1, img2])
        cv2.imwrite(f'{im_save_path}/{i}', three_c_img)



In [ ]:
#| export
def  get_common_fns(
        config:dict,
        training:bool=True
        ):
    'From config image and mask path get common files'
    if training:
        mask_names = set(get_name_(Path(config['dataset']['trn_msk_path']).ls()))
        images_names = set(get_name_(Path(config['dataset']['trn_im_path']).ls()))
    else:
        mask_names = set(get_name_(Path(config['dataset']['val_msk_path']).ls()))
        images_names = set(get_name_(Path(config['dataset']['val_im_path']).ls()))

    print(f' Number of mask files = {len(mask_names)}')
    print(f' Number of image files = {len(images_names)}')
    common_files = list(mask_names.intersection(images_names))
    print(f' Number of common files = {len(common_files)}')
    if training:
        ims = [Path(config['dataset']['trn_im_path'],i).as_posix() for i in common_files]
        msks = [Path(config['dataset']['trn_msk_path'], i).as_posix() for i in common_files]
    else:
        ims = [Path(config['dataset']['val_im_path'], i).as_posix() for i in common_files]
        msks = [Path(config['dataset']['val_msk_path'], i).as_posix() for i in common_files]
    return ims, msks

In [ ]:
#| export
def normalize(
              image:Union[np.ndarray, tf.Tensor], 
              min=0):
    def _normalize(im):
        img = tf.cast(im, tf.float32)
        return img / 255.0

    if min == 0:
        return _normalize(image)
    else:
        return (_normalize(image) * 2.0) -1.0

In [ ]:
#| export
def read_normalize(
                    im_file:str,
                    config:dict,
                    ):
    im = tf.io.read_file(im_file)
    im = tf.image.decode_png(
        im, 
        channels=config['dataset']['image_channels'])


    # in albumentation normalizing is done
    im = normalize(im,min=config['dataset']['min'])
    im = tf.cast(im,
                 tf.float32)
    
                        
    return im

In [ ]:
#| export
def read_and_binarize_mask(
                    mask_file:str,
                    config:dict,
                    ):
    msk = tf.io.read_file(mask_file)
    msk = tf.image.decode_png(
        msk, 
        channels=config['dataset']['mask_channels'])
    mask = tf.cast(msk > 127, tf.float32)
    return mask

In [ ]:
#| export
def process_image_and_mask(
                          img_path,
                          msk_path, 
                          config):
    image = read_normalize(
                           im_file=img_path,
                           config=config)
    mask = read_and_binarize_mask(
                                mask_file=msk_path,
                                config=config)
    return image, mask

In [ ]:
#| export
def augmentation_f(image, mask, config):
    # Static augmentation configuration
    aug_config = {
        'shift_limit': [0.04, 0.01],
        'scale_limit': [0.04, 0.01],
        'rotate_limit': [2, 5],
        'border_mode': cv2.BORDER_CONSTANT,  # Direct use of OpenCV constant
        'p_shift_scale_rotate': 0.4,
        'p_shift_only': 0.4,
        'p_horizontal_flip': 0.5,
        'p_vertical_flip': 0.5,
        #'p_transpose': 0.5,
        'p_brightness_contrast': 0.1,
        'p_color_jitter': 0.02,
        'p_gauss_noise': 0.2,
        'gauss_noise_var_limit': [0.01, 0.05],
        'p_gaussian_blur': 0.01,
        'gaussian_blur_limit': [3, 7],
        'p_multiplicative_noise': 0.3,
        'multiplicative_noise_multiplier': [0.05, 0.19]
    }

    im_h = config['dataset']['image_height']
    im_w = config['dataset']['image_width']


    # Map configuration to augmentation parameters
    shift_limit = tuple(aug_config['shift_limit'])
    scale_limit = tuple(aug_config['scale_limit'])
    rotate_limit = tuple(aug_config['rotate_limit'])
    border_mode = aug_config['border_mode']  # Assumes mapping to actual OpenCV constant done earlier
    p_shift_scale_rotate = aug_config['p_shift_scale_rotate']
    p_shift_only = aug_config['p_shift_only']
    p_horizontal_flip = aug_config['p_horizontal_flip']
    p_vertical_flip = aug_config['p_vertical_flip']
    #p_transpose = aug_config['p_transpose']
    p_brightness_contrast = aug_config['p_brightness_contrast']
    p_color_jitter = aug_config['p_color_jitter']
    p_gauss_noise = aug_config['p_gauss_noise']
    gauss_noise_var_limit = tuple(aug_config['gauss_noise_var_limit'])
    p_gaussian_blur = aug_config['p_gaussian_blur']
    gaussian_blur_limit = tuple(aug_config['gaussian_blur_limit'])
    p_multiplicative_noise = aug_config['p_multiplicative_noise']
    multiplicative_noise_multiplier = tuple(aug_config['multiplicative_noise_multiplier'])

    # Define the augmentation pipeline
    aug = A.Compose([
        A.OneOf([
            A.ShiftScaleRotate(
                shift_limit=shift_limit[0],
                scale_limit=scale_limit[0],
                rotate_limit=rotate_limit[0],
                border_mode=border_mode,
                p=p_shift_scale_rotate
    )]),
            #A.ShiftScaleRotate(
                #shift_limit=shift_limit[1],
                #scale_limit=scale_limit[1],
                #rotate_limit=rotate_limit[1],
                #border_mode=border_mode,
                #p=1.0 - p_shift_scale_rotate
            #)
        #], p=p_shift_scale_rotate),
        A.HorizontalFlip(p=p_horizontal_flip),
        A.VerticalFlip(p=p_vertical_flip),
        #A.Transpose(p=p_transpose),
        A.RandomBrightnessContrast(p=p_brightness_contrast),
        A.ColorJitter(brightness=0.7, p=p_color_jitter),
        A.GaussNoise(var_limit=gauss_noise_var_limit, p=p_gauss_noise),
        A.GaussianBlur(blur_limit=gaussian_blur_limit, p=p_gaussian_blur),
        A.RandomResizedCrop(
            height=im_h, 
            width=im_w, 
            scale=(0.8, 1.0), 
            ratio=(0.75, 1.33), 
            p=0.5),
        #A.Resize(1152,1632, p=1.0),
        #A.MultiplicativeNoise(
            #multiplier=multiplicative_noise_multiplier, 
            #per_channel=True, 
            #p=p_multiplicative_noise
        #),
    ])

    # Apply the augmentations
    aug_data = aug(image=image, mask=mask)
    image, mask = aug_data['image'], aug_data['mask']

    return image, mask

In [ ]:
#| export
def augment_data(
                image, 
                mask, 
                config):
    aug_config = config['dataset']['augmentations']
    data_config = config['dataset']
    image_shape = (
                    data_config['image_height'], 
                    data_config['image_width'], 
                    data_config['image_channels']
                 )

    mask_shape = (
                    data_config['image_height'], 
                    data_config['image_width'], 
                    data_config['mask_channels']
                 )

    aug_func = partial(augmentation_f, config=config)

    aug_img, aug_mask = tf.numpy_function(
        func=aug_func,
        inp=[
            image,
            mask
        ],
        Tout=(tf.float32, tf.float32)
    )

    aug_img.set_shape(image_shape)
    aug_mask.set_shape(mask_shape)

    print(f'Augmented image shape = {aug_img.shape}') 
    print(f'Augmented image shape = {aug_mask.shape}') 

    #aug_img = tf.image.resize(
        #aug_img, 
        #[data_config['image_height'], 
         #data_config['image_width']])
    #aug_mask = tf.image.resize(
        #aug_mask [data_config['image_height'], data_config['image_width']])


    aug_mask = tf.cast(aug_mask > 0.5, tf.float32)

    return aug_img, aug_mask

In [ ]:
#| export
def mixup_segmentation(
        images, 
        masks, 
        ALPHA=0.2):
    ' Mix up Augmentation following beta dist'
    if ALPHA <= 0:
        return images, masks

    batch_size = tf.shape(images)[0]
    weights = tf.random.uniform((batch_size, 1, 1, 1), minval=0, maxval=1)
    mix_weights = tf.maximum(weights, 1 - weights)  # Ensure symmetry

    # Randomly permute the batch to mix
    perm = tf.random.shuffle(tf.range(batch_size))
    perm_images = tf.gather(images, perm, axis=0)
    perm_masks = tf.gather(masks, perm, axis=0)

    mixed_images = mix_weights * images + (1 - mix_weights) * perm_images
    mixed_masks = mix_weights * masks + (1 - mix_weights) * perm_masks

    return mixed_images, mixed_masks

In [ ]:
#| export
def cutmix_segmentation(
        images, 
        masks, 
        PROBABILITY=.1):
    # Assumes images and masks are in the same batch and aligned
    if tf.random.uniform([]) > PROBABILITY:
        return images, masks

    batch_size, height, width, _ = images.shape
    mixing_images = images
    mixing_masks = masks



    # Randomly choose another element in the batch
    perm = tf.random.shuffle(tf.range(batch_size))
    perm_images = tf.gather(images, perm, axis=0)
    perm_masks = tf.gather(masks, perm, axis=0)

    # Randomly choose the mix ratio and the position to cut
    cut_rat = tf.sqrt(1 - tf.random.uniform([]) * 0.3)
    cut_h = tf.cast(height * cut_rat, tf.int32)
    cut_w = tf.cast(width * cut_rat, tf.int32)

    cx = tf.random.uniform([], int(width * 0.1), int(width * 0.9), tf.int32)
    cy = tf.random.uniform([], int(height * 0.1), int(height * 0.9), tf.int32)

    ymin = tf.clip_by_value(cy - cut_h // 2, 0, height)
    ymax = tf.clip_by_value(cy + cut_h // 2, 0, height)
    xmin = tf.clip_by_value(cx - cut_w // 2, 0, width)
    xmax = tf.clip_by_value(cx + cut_w // 2, 0, width)

    # Create masks that will be used to slice and mix the images
    mask1 = tf.concat([
        tf.ones((batch_size, ymin, width, 1)),
        tf.concat([
            tf.ones((batch_size, ymax - ymin, xmin, 1)),
            tf.zeros((batch_size, ymax - ymin, xmax - xmin, 1)),
            tf.ones((batch_size, ymax - ymin, width - xmax, 1))
        ], axis=2),
        tf.ones((batch_size, height - ymax, width, 1))
    ], axis=1)

    images = mask1 * images + (1 - mask1) * perm_images
    masks = mask1 * masks + (1 - mask1) * perm_masks

    return images, masks

In [ ]:
#| export
def create_dataset_w_mix(
        config:dict,
        training:bool=True,
        only_cutmix:bool=False,
        only_mixup:bool=False,
        both:bool=True
    ):
    ims, msks = get_common_fns(config,training=training)
    dataset = tf.data.Dataset.from_tensor_slices((ims, msks))
    dataset = dataset.map(
        partial(process_image_and_mask, config=config),
        num_parallel_calls=tf.data.experimental.AUTOTUNE
    )
    if training:
        dataset = dataset.map(
            partial(augment_data, config=config),
            num_parallel_calls=tf.data.experimental.AUTOTUNE
        )

        dataset = dataset.shuffle(config['dataset']['buffer_size'])

    dataset = dataset.batch(
        config['training']['batch_size'],
        drop_remainder=True)

    if training:
        if only_cutmix:
            dataset = dataset.map(
                lambda image, mask: cutmix_segmentation(image, mask, PROBABILITY=0.5),
                num_parallel_calls=tf.data.experimental.AUTOTUNE
            )

        elif only_mixup:
            dataset = dataset.map(
                lambda image, mask: mixup_segmentation(image, mask, ALPHA=0.2),
                num_parallel_calls=tf.data.experimental.AUTOTUNE
            )
        elif both:
            dataset = dataset.map(
                lambda image, mask: cutmix_segmentation(image, mask, PROBABILITY=0.5),
                num_parallel_calls=tf.data.experimental.AUTOTUNE
            )
            dataset = dataset.map(
                lambda image, mask: mixup_segmentation(image, mask, ALPHA=0.2),
                num_parallel_calls=tf.data.experimental.AUTOTUNE
            )
        else: dataset = dataset
    dataset = dataset.repeat()
    dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)
    return dataset



In [ ]:
#| export
def create_dataset(
        config:dict,
        training:bool=True,
    ):
    ims, msks = get_common_fns(config,training=training)
    dataset = tf.data.Dataset.from_tensor_slices((ims, msks))
    dataset = dataset.map(
        partial(process_image_and_mask, config=config),
        num_parallel_calls=tf.data.experimental.AUTOTUNE
    )
    if training:
        dataset = dataset.map(
            partial(augment_data, config=config),
            num_parallel_calls=tf.data.experimental.AUTOTUNE
        )
        dataset = dataset.shuffle(config['dataset']['buffer_size'])
    dataset = dataset.repeat()
    dataset = dataset.batch(config['training']['batch_size'])
    dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)
    return dataset



In [ ]:
#| export
def create_color_mask(
           
             mask:Union[np.array, tf.Tensor], 
             img:Union[np.array, tf.Tensor],
             config:dict,
             threshold:float=0.5):
    'Creating color mask for segmentation'

    im_height=config['dataset']['image_height']
    im_width=config['dataset']['image_width']
    
    overlay_mask = np.ones((im_height, im_width, 3,), dtype=np.uint8)
    overlay_mask[:, :, 0] = img.reshape(*(im_height, im_width))
    overlay_mask[:, :, 1] = img.reshape(*(im_height, im_width))
    overlay_mask[:, :, 2] = img.reshape(*(im_height, im_width))
    match = mask.reshape(*(im_height, im_width)) > (threshold * 255)
    overlay_mask[match] = [0,255,0]
    return overlay_mask


In [ ]:
#| export
def display_np_batch(
                  
                    images:np.ndarray,
                    masks:np.ndarray,
                    config:dict,
                   no_img=5,
                    threshold:float=0.5):
    im_height=config['dataset']['image_height']
    im_width=config['dataset']['image_width']
    "Displaying batch of images and masks"
    n_batch = images.shape[0]
    no_img_ = np.min([no_img, n_batch])
    for i in range(no_img_):
        if images[i].shape[2] ==3:
            img = images[i][:,:,0]
            mask = masks[i][:,:,0]
        else:
            img = images[i].reshape(im_height, im_width )
            mask = masks[i].reshape(im_height, im_width )
        
        _, ax = plt.subplots(1, 3, figsize=(15, 10)) 
        ax[0].imshow(img, cmap='gray')
        ax[0].axis('off')
        ax[0].set_title('only image')
        clr_mask_ = create_color_mask(mask=mask, img=img, threshold=threshold, config=config)
        ax[1].imshow(mask)
        ax[1].axis('off')
        ax[1].set_title('only mask')
        ax[2].imshow(clr_mask_)
        ax[2].axis('off')
        ax[2].set_title('image with mask')
        plt.tight_layout()

In [ ]:
#| export
def convert_np_and_uint8(img:tf.Tensor)->Tuple[np.array, np.array]:
    "Convert img to np.array and uint8"

    if isinstance(img, tf.data.Dataset) or isinstance(img, tf.Tensor):
        m_scale1=img.numpy()
        m_scale255=(img * 255).numpy().astype(np.uint8)
    elif isinstance(img, np.ndarray):
            
        if img.dtype in [np.float16, np.float32, np.float64]:
            m_scale255 = (img * 255).astype(np.uint8)
            m_scale1 = img
            raise Exception("unknown dtype:", img.dtype)

    return m_scale1, m_scale255


In [ ]:
#| export
def display_ds(
            ds:tf.data.Dataset,
            config:dict,
            threshold:float,
            no_img:int):

    images, masks = next(iter(ds))

    # convert to numpy and uint8 and getting scaled(255) image and mask
    images_np, images_255_np = convert_np_and_uint8(images)
    masks_np, masks_255_np = convert_np_and_uint8(masks)
    display_np_batch(
                          images=images_255_np,
                          masks=masks_255_np, 
                          config=config,no_img=no_img,
                          threshold=threshold)

In [ ]:
#display_ds(trn_dataset, config, threshold=0.5, no_img=5)

In [ ]:
# Assuming `dataset` is your TensorFlow dataset
def visualize_random_batch(dataset, batch_size=4):
    # Get a random batch from the dataset
    random_batch = next(iter(dataset.shuffle(buffer_size=10).batch(batch_size)))

    # Split images and masks
    images, masks = random_batch

    # Visualize the batch
    visualize_batch(images.numpy(), masks.numpy())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('10_three_channel_dataset_creation.ipynb')